# 01 — H₂ All Methods Benchmark

Compare **all classical and quantum solvers** on H₂ (STO-3G) at the equilibrium geometry.

| Method | Type | Scaling |
|--------|------|---------|
| HF     | Classical | O(N³) |
| MP2    | Classical | O(N⁵) |
| CISD   | Classical | O(N⁶) |
| CCSD   | Classical | O(N⁶) |
| FCI    | Classical (exact) | O(exp N) |
| VQE-UCCSD | Quantum | — |
| VQE-HEA   | Quantum | — |
| ADAPT-VQE | Quantum | — |
| QPE (ideal) | Quantum | — |
| SQD   | Quantum+Classical | — |


In [1]:
import sys
from pathlib import Path

# Add package root if needed
pkg_root = Path.cwd().parent
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))

# Register all solvers
import quantum_chem_bench.classical_solvers  # noqa
import quantum_chem_bench.quantum_solvers    # noqa

from quantum_chem_bench.core.config import BenchConfig
from quantum_chem_bench.core.runner import BenchRunner
from quantum_chem_bench.analysis.benchmark import BenchmarkPlotter, format_table

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'quantum_chem_bench'

## Load config and run all solvers

In [ ]:
config = BenchConfig.from_yaml('../configs/h2_sto3g.yaml')

print(f"Molecule: {config.molecule.geometry}")
print(f"Basis:    {config.molecule.basis}")
print(f"Active space: {config.molecule.n_active_electrons} electrons, "
      f"{config.molecule.n_active_orbitals} orbitals")
print(f"\nClassical solvers: {config.solvers.classical}")
print(f"Quantum solvers:   {config.solvers.quantum}")

In [ ]:
# CI_FAST_NB=1 skips expensive quantum solvers for quick notebook smoke-tests
import os
if os.environ.get('CI_FAST_NB'):
    config.solvers.quantum = ['vqe_uccsd']

runner = BenchRunner(config, verbose=True)
bench = runner.run()

## Results table

In [ ]:
df = pd.DataFrame(bench.summary_table())
print(f"FCI energy (exact reference): {bench.fci_energy:.10f} Ha")
print(f"HF  energy (mean field):      {bench.hf_energy:.10f} Ha\n")
df

## Energy error bar chart vs FCI

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

plotter = BenchmarkPlotter(figsize=(10, 5))
fig = plotter.energy_bar(
    bench,
    reference='fci',
    title='H₂ (STO-3G) — All Methods Energy Error vs FCI',
)
fig.savefig('h2_energy_errors.png', dpi=150)
fig

## Swap solver with one line

The registry pattern allows swapping any solver instantly:

In [ ]:
from quantum_chem_bench.core.registry import registry
print('Registered solvers:', registry.list_names(category='solver'))

In [ ]:
# Run just one solver programmatically
from quantum_chem_bench.core.interfaces import MolSpec

mol = MolSpec(geometry='H 0 0 0; H 0 0 0.735', basis='sto-3g',
              n_active_electrons=(1,1), n_active_orbitals=2)

for name in ['hf', 'ccsd', 'fci']:
    solver = registry.build(name, category='solver')
    r = solver.solve(mol)
    print(f'{name.upper():10s}  E = {r.energy:+.10f} Ha  ({r.wall_time:.2f}s)')